# 01 — Load a GMNS network

GMNS (General Modeling Network Specification) is the open network format TAPLite4MPO reads. A scenario folder is a handful of CSVs. This notebook opens the **Chicago Sketch** network through the API and explains each piece: **nodes, links, zones, demand**.

In [1]:
from dtalite_qa.api import Network, Demand
net = Network.read_gmns('kernel/data_sets/03_chicago_sketch')
net.summary()

{'nodes': 933,
 'links': 2950,
 'zones': 387,
 'folder': 'C:\\source_codes\\0_source_code_new\\dtalite_with_taplite_Cpp_kernel\\kernel\\data_sets\\03_chicago_sketch'}

## Nodes — `node.csv`
Every intersection / centroid. A node with `zone_id > 0` is a **centroid**: an origin/destination of demand. The rest are physical junctions.

In [2]:
import csv, itertools, os
with open(os.path.join(net.folder, 'node.csv'), encoding='utf-8-sig') as f:
    rows = list(itertools.islice(csv.DictReader(f), 5))
print('columns:', list(rows[0].keys()))
for r in rows: print({k: r[k] for k in ('node_id','zone_id','x_coord','y_coord')})
print('total nodes:', net.n_nodes, '| zones (centroids):', net.n_zones)

columns: ['node_id', 'zone_id', 'x_coord', 'y_coord']
{'node_id': '1', 'zone_id': '1', 'x_coord': '-87.67559107', 'y_coord': '42.01165238'}
{'node_id': '2', 'zone_id': '2', 'x_coord': '-87.69940207', 'y_coord': '42.00355037'}
{'node_id': '3', 'zone_id': '3', 'x_coord': '-87.66487612', 'y_coord': '41.97744392'}
{'node_id': '4', 'zone_id': '4', 'x_coord': '-87.68987767', 'y_coord': '41.96394058'}
{'node_id': '5', 'zone_id': '5', 'x_coord': '-87.65297062', 'y_coord': '41.93243279'}
total nodes: 933 | zones (centroids): 387


## Links — `link.csv`
Directed road segments. The fields that drive the assignment:
- `from_node_id → to_node_id` — topology
- `lanes`, `capacity` — supply (capacity is **per-lane**)
- `free_speed` / `vdf_free_speed_mph`, `length` / `vdf_length_mi` — free-flow time
- `vdf_alpha`, `vdf_beta`, `vdf_type` — the volume-delay function (BPR here)
- `allowed_uses` — mode access control
- `ref_volume` — a reference/count column used for validation

In [3]:
with open(os.path.join(net.folder, 'link.csv'), encoding='utf-8-sig') as f:
    r0 = next(csv.DictReader(f))
keys = ['link_id','from_node_id','to_node_id','lanes','capacity',
        'vdf_free_speed_mph','vdf_alpha','vdf_beta','ref_volume']
for k in keys: print(f'{k:20} {r0.get(k)}')
print('total links:', net.n_links)

link_id              1
from_node_id         1
to_node_id           547
lanes                1
capacity             49500
vdf_free_speed_mph   60
vdf_alpha            0.15
vdf_beta             4
ref_volume           4989.13
total links: 2950


## Zones & demand — `demand.csv`
OD demand is `o_zone_id, d_zone_id, volume`. `volume` is trips over the analysis period; whether those are **vehicles or persons** is a *declared* convention (the intake gate blocks if undeclared — it must not be guessed).

In [4]:
d = Demand.from_network(net)
print('demand files:', d.files)
with open(os.path.join(net.folder, 'demand.csv'), encoding='utf-8-sig') as f:
    for r in itertools.islice(csv.DictReader(f), 4): print(r)
print('total OD trips:', f'{d.total_trips():,.0f}')

demand files: ['demand.csv']
{'o_zone_id': '1', 'd_zone_id': '1', 'volume': '273.18'}
{'o_zone_id': '1', 'd_zone_id': '2', 'volume': '347.31'}
{'o_zone_id': '1', 'd_zone_id': '3', 'volume': '390.81'}
{'o_zone_id': '1', 'd_zone_id': '4', 'volume': '204.51'}


total OD trips: 1,260,907


## The declared conventions — `submission.yml`
The one file a shapefile + matrix *cannot* carry. Chicago Sketch ships with it filled, so the gate is READY.

In [5]:
print(open(os.path.join(net.folder, 'submission.yml'), encoding='utf-8').read())

# =============================================================================
# Chicago Sketch — the in-repo teaching / performance network.
# This is a native GMNS scenario shipped with the kernel (not an agency hand-off),
# so the conventions are known and declared here so the no-guessing intake gate
# passes cleanly. It is the reference example an MPO copies for their own data.
#   Run:  taplite validate kernel/data_sets/03_chicago_sketch
# =============================================================================
agency: TAPLite4MPO sample (Chicago Sketch, classic Bar-Gera testbed)
model_year: 1990
contact: TAPLite4MPO maintainers

# --- CAPACITY ---------------------------------------------------------------
capacity_basis: per_lane          # link.csv 'capacity' is per-lane (DTALite convention)
capacity_period: hourly           # capacity is an hourly LOS-E capacity
capacity_source_field: capacity
capacity_period_hours: 1

# --- TIME PERIOD & PEAK LOAD FACTOR ---------------

In [6]:
from dtalite_qa import control
ok, status, reason = control.check_intake_gate(net.folder)
print('intake gate:', status, '| ok to run:', ok)

intake gate: READY | ok to run: True


Next: **02_baseline_assignment** — run the equilibrium and read the report.